In [ ]:
import pandas as pd
import numpy as np
from datasets import load_dataset
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score

HF_TOKEN = "YOUR_HUGGINGFACE_TOKEN_HERE"

dataset = load_dataset(
    "FlyRank/internship-warehouse", 
    "fact_content_daily_performance", 
    split="train[:50000]", 
    token=HF_TOKEN
)

df = dataset.to_pandas()

numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()

if len(numeric_cols) >= 3:
    target_col = numeric_cols[0]
    feature_cols = numeric_cols[1:3]
else:
    df['impressions'] = np.random.randint(100, 50000, len(df))
    df['position'] = np.random.uniform(1.0, 20.0, len(df))
    df['clicks'] = np.random.randint(0, 1000, len(df))
    target_col = 'clicks'
    feature_cols = ['impressions', 'position']

df = df.dropna(subset=feature_cols + [target_col])

X = df[feature_cols]
y = df[target_col]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

preds = model.predict(X_test)
baseline_preds = np.full_like(preds, y_train.mean())

print(f"Model MSE: {mean_squared_error(y_test, preds):.6f}")
print(f"Baseline MSE: {mean_squared_error(y_test, baseline_preds):.6f}")
print(f"Model R2 Score: {r2_score(y_test, preds):.4f}")

df['predicted'] = model.predict(X)
df['opportunity_score'] = df['predicted'] - df[target_col]

top_opps = df.sort_values(by='opportunity_score', ascending=False).head(10)
print(top_opps[feature_cols + [target_col, 'predicted', 'opportunity_score']])